# Extracción de datos - Telecom X

Este notebook extrae los datos de la API de Telecom X en formato JSON y los carga en un DataFrame de Pandas, aplanando las columnas anidadas (como `customer`, `internet`, y `account`) para facilitar el análisis.

In [ ]:
import pandas as pd
import requests
import numpy as np

In [ ]:
# URL de los datos brutos en GitHub
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"

print("Descargando datos...")
response = requests.get(url)

# Verificar respuesta
if response.status_code == 200:
    data = response.json()
    
    # Aplanar el JSON y crear el DataFrame
    df = pd.json_normalize(data)
    
    print("\n¡Datos cargados con éxito!")
    print(f"Total de clientes (filas): {df.shape[0]}")
    print(f"Total de atributos (columnas): {df.shape[1]}")
else:
    print(f"Error: {response.status_code}")

In [ ]:
# Mostrar 5 primeras filas
df.head()

## 1. Exploración de la Estructura de los Datos
El primer paso fundamental es comprender qué tipos de datos tenemos y si hay valores nulos. Para esto, utilizaremos los métodos `df.info()` y `df.dtypes`.

In [ ]:
# Información General del DataFrame (Filas, Columnas, Nulos y Tipos generales)
print("--- Información del DataFrame ---")
df.info()

print("\n\n--- Tipos de Datos (dtypes) ---")
display(df.dtypes)

## Diccionario de Datos y Relevancia para la Evasión
Basado en el contexto de Telecom X, estas son las variables clave y su potencial impacto en la evasión (`Churn`):

### Variable Objetivo
- **`Churn`**: Variable categórica (Yes/No) que indica si el cliente abandonó la empresa en el último mes. **Esta es la variable más importante**, ya que es lo que queremos predecir.

### Columnas del Cliente (Demográficas)
- **`customer.tenure`**: **MUY IMPORTANTE.** Meses de permanencia en la empresa. Los clientes nuevos suelen tener tasas de evasión más altas que los clientes leales de años.
- **`customer.SeniorCitizen`**: Indica si el cliente es mayor (1) o no (0).
- **`customer.Partner` / `customer.Dependents`**: Si el cliente tiene pareja o dependientes.
- **`customer.gender`**: Género del cliente. Suele tener impacto bajo.

### Columnas de Cuenta y Cargos
- **`account.Contract`**: **MUY IMPORTANTE.** Tipo de contrato. Los contratos mes a mes casi siempre tienen el mayor índice de evasión.
- **`account.Charges.Monthly`**: Cargo mensual. **Crítico.** Aumento en costos es causa principal de abandono.
- **`account.Charges.Total`**: Cargo total histórico. Aquí debes notar en `df.info()` que fue interpretado como un objeto (string) en lugar de número, indicando necesidad de limpieza.

## 2. Verificación Continua y Limpieza
Buscar problemas en los datos que puedan afectar el análisis, como valores ausentes (NaN) o filas duplicadas.

In [ ]:
# 1. Verificar valores ausentes (Nulos reales)
nulos_totales = df.isnull().sum()
print("Valores nulos en cada columna:")
print(nulos_totales[nulos_totales > 0]) # Solo mostramos las que tienen nulos técnicos

# 2. Verificar filas duplicadas
duplicados = df.duplicated().sum()
print(f"\nNúmero de filas exactamente duplicadas: {duplicados}")

## 3. Limpieza de Errores de Formato
Hemos detectado que `account.Charges.Total` es tipo `object` debido a espacios vacíos, y que nuestra variable fundamental `Churn` también tiene registros vacíos.

In [ ]:
print("--- LIMPIEZA DE DATOS ---")

# PASO 1: Arreglar account.Charges.Total
vacios_cargos = df[df['account.Charges.Total'] == ' '].shape[0]
print(f"Cargos Totales con espacios en blanco encontrados: {vacios_cargos}")

df['account.Charges.Total'] = df['account.Charges.Total'].replace(' ', np.nan)
df['account.Charges.Total'] = pd.to_numeric(df['account.Charges.Total'])
df['account.Charges.Total'] = df['account.Charges.Total'].fillna(0)
print("1. Columna account.Charges.Total corregida y convertida a tipo numérico.")

# PASO 2: Arreglar la variable objetivo (Churn)
churn_vacios = df[df['Churn'] == ""].shape[0]
print(f"Registros de Churn sin valor (vacíos o espacios): {churn_vacios}")

df['Churn'] = df['Churn'].replace([' ', ''], np.nan)
df = df.dropna(subset=['Churn'])
print("2. Filas con 'Churn' nulo eliminadas, ya que no sirven para entrenar el modelo.")

# Verificación final
print("\n--- RESULTADO DE LA LIMPIEZA ---")
print(f"Nuevas dimensiones: {df.shape}")

In [ ]:
# Validación de que Charges.Total ahora es float64
df.info()

## 4. Manipulación y Estandarización de Textos (Strings)

Tras limpiar los errores numéricos básicos, es vital asegurar que los datos de tipo texto (`object`) sean consistentes. Utilizaremos el accesor `.str` de Pandas para modificar cadenas de texto. Esto evita que categorías como 'Yes' y 'yes' sean tratadas como diferentes por un modelo. Ver el [Tutorial de Alura](https://www.aluracursos.com/blog/manipulacion-de-strings-en-pandas-lower-replace-startswith-y-contains).

In [ ]:
print("--- ESTANDARIZACIÓN DE STRINGS ---")

# 1. Convertir nombres de columnas problemáticas (si las hay)
print("Nombres de columnas originales (primeros 5):")
print(df.columns[:5].tolist())

# Reemplazar los puntos '.' que dejamos del json_normalize por guiones bajos '_'
# usando .str.replace()
df.columns = df.columns.str.replace('.', '_')
print("\nNuevos nombres de columnas (usando replace):")
print(df.columns[:5].tolist())

In [ ]:
# 2. Estandarizar valores categóricos a Minúsculas con .str.lower()
# Esto asegura que evitar inconsistencias humanas.

# Vamos a aplicarlo a la columna objetivo, 'Churn'.
df['Churn'] = df['Churn'].str.lower()
print("Valores en 'Churn' después de aplicar lower():")
print(df['Churn'].unique())

In [ ]:
# 3. Limpieza de descripciones usando .str.replace()

print("Métodos de pago originales:")
print(df['account_PaymentMethod'].unique())

# Algunos dicen '(automatic)', lo reemplazaremos por nada (lo eliminamos)
# Usamos regex=True para manejar expresiones regulares; '\(' escapa los paréntesis.
df['account_PaymentMethod'] = df['account_PaymentMethod'].str.replace(' \(automatic\)', '', regex=True)

print("\nMétodos de pago limpios (sin '(automatic)'):")
print(df['account_PaymentMethod'].unique())

In [ ]:
# 4. Búsquedas e identificaciones usando .str.startswith() y .str.contains()

# STARTWITH: Identificar los contratos que comiencen por la cadena "Month"
contratos_mensuales = df[df['account_Contract'].str.startswith('Month')]
print(f"Clientes con contrato que inicia en 'Month': {contratos_mensuales.shape[0]}")

# CONTAINS: Identificar aquellos servicios de internet que de alguna forma contengan "Fiber"
clientes_fibra = df[df['internet_InternetService'].str.contains('Fiber', case=False, na=False)]
print(f"Clientes con internet de fibra (usando contiene): {clientes_fibra.shape[0]}")

## 5. Ingeniería de Características (Feature Engineering): Nuevas Variables

Para obtener una vista más detallada del comportamiento diario de facturación, calcularemos el gasto diario esperado por cada cliente basándonos en su factura mensual. Esto nos brindará una granularidad distinta al analizar la evasión.

In [ ]:
# Calculamos 'Cuentas_Diarias' dividiendo el cargo mensual ('account_Charges_Monthly') entre 30 días.
# Usamos round() para mantener 2 decimales y que sea legible como moneda.
df['Cuentas_Diarias'] = round(df['account_Charges_Monthly'] / 30, 2)

print("Primeras filas mostrando los cargos mensuales y la nueva columna diaria:")
display(df[['account_Charges_Monthly', 'Cuentas_Diarias']].head())

## 6. Estandarización y Transformación de Datos

Para preparar los datos para los modelos analíticos y hacerlos más interpretables, realizaremos dos tareas clave:
1. **Binarización**: Convertir los valores de texto afirmativos/negativos ('yes', 'no', 'Yes', 'No') en variables numéricas binarias (1 y 0). Los algoritmos matemáticos procesan mejor números que texto.
2. **Renombrar Columnas**: Traducir los nombres técnicos en inglés a español, lo cual facilitará la interpretación para los stakeholders (partes interesadas) no técnicos.

In [ ]:
# 1. Binarización de Variables Categóricas (Yes/No a 1/0)

# Usamos un diccionario de mapeo
mapeo_binario = {'yes': 1, 'no': 0, 'Yes': 1, 'No': 0}

# Identificamos las columnas que contienen típicamente Yes/No
columnas_binarias = [
    'Churn', 'customer_Partner', 'customer_Dependents', 
    'phone_PhoneService', 'account_PaperlessBilling'
]

print("Transformando 'Yes' a 1 y 'No' a 0...")
# Aplicamos el mapeo a las columnas seleccionadas
for col in columnas_binarias:
    if col in df.columns:
        # Usamos map() para reemplazar los valores según el diccionario
        # Si existe algún valor distinto a 'yes'/'no' se convertirá en NaN, 
        # lo cual es útil para detectar inconsistencias ocultas.
        df[col] = df[col].map(mapeo_binario).fillna(df[col])

print("\nValores únicos en 'Churn' después de binarizar:")
print(df['Churn'].unique())

In [ ]:
# 2. Renombrar Columnas al Español para Claridad

traduccion_columnas = {
    'customerID': 'ID_Cliente',
    'Churn': 'Evasion',
    'customer_gender': 'Genero',
    'customer_SeniorCitizen': 'Adulto_Mayor',
    'customer_Partner': 'Tiene_Pareja',
    'customer_Dependents': 'Tiene_Dependientes',
    'customer_tenure': 'Meses_Permanencia',
    'phone_PhoneService': 'Servicio_Telefonico',
    'phone_MultipleLines': 'Multiples_Lineas',
    'internet_InternetService': 'Servicio_Internet',
    'internet_OnlineSecurity': 'Seguridad_Online',
    'internet_OnlineBackup': 'Respaldo_Online',
    'internet_DeviceProtection': 'Proteccion_Dispositivo',
    'internet_TechSupport': 'Soporte_Tecnico',
    'internet_StreamingTV': 'TV_Streaming',
    'internet_StreamingMovies': 'Peliculas_Streaming',
    'account_Contract': 'Tipo_Contrato',
    'account_PaperlessBilling': 'Factura_Electronica',
    'account_PaymentMethod': 'Metodo_Pago',
    'account_Charges_Monthly': 'Cargo_Mensual',
    'account_Charges_Total': 'Cargo_Total'
}

# Aplicamos el renombramiento
df = df.rename(columns=traduccion_columnas)

print("Nuevos nombres de columnas (Español):")
print(df.columns.tolist())

print("\n--- MUESTRA FINAL DEL DATASET ESTANDARIZADO ---")
display(df.head())

## 7. Análisis Descriptivo

El análisis descriptivo nos ayuda a entender las distribuciones matemáticas de nuestros datos limpios y estandarizados. Utilizaremos el método `describe()` de Pandas, el cual calcula estadísticas de resumen como:

- **Media (mean):** El promedio de los valores.
- **Desviación Estándar (std):** Cuánto varían los datos respecto al promedio.
- **Mínimo y Máximo (min, max):** Los valores extremos.
- **Cuartiles (25%, 50%, 75%):** El 50% es la Mediana. Resulta útil para observar sesgos.

In [ ]:
print("--- ESTADÍSTICAS DESCRIPTIVAS (Variables Numéricas) ---")

# describe() por defecto solo toma las columnas numéricas (int64, float64).
# Gracias a nuestro trabajo previo, variables como 'Cargo_Total' y 'Cuentas_Diarias'
# ya aparecerán aquí correctamente calculadas.

estadisticas = df.describe().round(2) # Redondeamos a 2 decimales para leerlo mejor
display(estadisticas)

In [ ]:
print("--- ANÁLISIS DE LA VARIABLE OBJETIVO (Evasión) ---")

# Como convertimos la Evasión ('Churn') de Yes/No a 1/0, 
# ahora podemos calcular su MEDIA, lo que nos da la tasa exacta de abandono global.

tasa_evasion = round(df['Evasion'].mean() * 100, 2)
print(f"La tasa global de evasión es del {tasa_evasion}%")

### Qué observar en la tabla `describe()`:
1. **`Adulto_Mayor` / `Evasion`**: Al ser variables binarias (0 y 1), su media indica la proporción (ej. si la media de `Evasion` es 0.26, significa que el ~26% de los clientes se fue).
2. **`Meses_Permanencia`**: La divergencia entre la media y la mediana (el valor del 50%) te dirá si tienes más clientes antiguos o recientes.
3. **`Cargo_Mensual` vs `Cuentas_Diarias`**: Podrás ver el promedio de lo que pagan tus clientes y cuáles son sus extremos y desviaciones.

## 8. Análisis de la Variable Objetivo (Evasión / Churn)

El primer paso en la exploración visual es entender nuestro problema principal: ¿Cuántos clientes se están yendo frente a los que se quedan? 
Para esto, utilizaremos las librerías `matplotlib` y `seaborn` para crear gráficos de barras y de pastel.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual general para que los gráficos se vean modernos y limpios
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
colores = ["#2ecc71", "#e74c3c"] # Verde (Permanece), Rojo (Evasión)

In [ ]:
print("--- DISTRIBUCIÓN DE EVASIÓN (CHURN) ---")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Análisis de Retención vs Evasión de Clientes", fontsize=16, fontweight='bold')

# GRÁFICO 1: Gráfico de barras (Conteo absoluto)
sns.countplot(data=df, x='Evasion', palette=colores, ax=axes[0])
axes[0].set_title("Cantidad de Clientes por Estado", fontsize=14)
axes[0].set_xlabel("Evasión (0 = No, 1 = Sí)", fontsize=12)
axes[0].set_ylabel("Número de Clientes", fontsize=12)

# Agregar las etiquetas de datos sobre las barras
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha = 'center', va = 'bottom', 
                     fontsize=11)
    
# GRÁFICO 2: Gráfico de Pastel (Proporción relativa)
evasion_counts = df['Evasion'].value_counts()
axes[1].pie(evasion_counts, 
            labels=["Permanecen (0)", "Se dan de baja (1)"], 
            autopct='%1.1f%%', 
            startangle=90, 
            colors=colores, 
            explode=(0, 0.1),  # Separar un poco el 'Sí' para resaltarlo
            shadow=True)
axes[1].set_title("Proporción de Evasión", fontsize=14)

plt.tight_layout()
plt.show()

### Conclusiones del Gráfico:
Este gráfico de pastel y barras revela el desbalance de nuestro dataset. Notarás que la proporción de personas que se han ido (alrededor del 26%) es considerablemente menor a las que se quedan. Esto se llama **clases desbalanceadas** en Machine Learning, algo vital de considerar al entrenar modelos predictivos para que no se sesguen a predecir siempre que el cliente se queda.

## 9. Análisis de Evasión por Variables Categóricas

Ahora cruzaremos nuestra variable objetivo (`Evasion`) contra variables categóricas clave (Tipo de Contrato, Servicio de Internet, Método de Pago, etc.) para buscar patrones. 

Utilizaremos `sns.countplot` separando los colores mediante el parámetro `hue="Evasion"`.

In [ ]:
print("--- EVASIÓN POR PERFIL DEL CLIENTE ---")

# Variables a analizar
columnas_cat = ['Tipo_Contrato', 'Servicio_Internet', 'Metodo_Pago', 'Adulto_Mayor']
nombres_titulos = ['Tipo de Contrato', 'Servicio de Internet', 'Método de Pago', 'Adulto Mayor (1=Sí)']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Patrones de Evasión según Características del Servicio y Cliente", fontsize=18, fontweight='bold', y=1.02)

axes = axes.flatten() # Para iterar fácilmente sobre la grilla 2x2

for i, col in enumerate(columnas_cat):
    sns.countplot(data=df, x=col, hue='Evasion', palette=colores, ax=axes[i])
    axes[i].set_title(f"Evasión por {nombres_titulos[i]}", fontsize=14)
    axes[i].set_xlabel("") # Limpiamos el texto del eje X por estética
    axes[i].set_ylabel("Número de Clientes")
    axes[i].legend(title="Evasión (1=Sí)")
    
    # Si es el método de pago, rotamos las etiquetas para que se lean bien
    if col == 'Metodo_Pago':
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 💡 Análisis Estratégico (Insights):
Al observar estos gráficos, los patrones estratégicos se hacen evidentes de inmediato:
1. **Tipo de Contrato:** El contrato "Month-to-month" (mes a mes) es absolutamente tóxico para la retención. La gran mayoría de las fugas provienen de aquí. Los contratos de 1 y 2 años tienen fugas casi nulas en comparación.
2. **Servicio de Internet:** Los clientes de **Fibra Óptica** tienen una tasa de abandono preocupantemente alta en contraste con el ADSL. Esto sugiere investigar fallas en el servicio, precios demasiado altos de la fibra, o competidores más agresivos en ese segmento.
3. **Método de Pago:** Los clientes que pagan con "Electronic check" (Cheque electrónico) son los más propensos a irse.
4. **Adultos Mayores:** Aunque son un grupo pequeño (1), su proporción de evasión es visiblemente mayor comparada con los clientes jóvenes (0). Podrían estar experimentando barreras tecnológicas o problemas de soporte.

## 10. Análisis de Evasión por Variables Numéricas

Para entender cómo los números continuos (dinero y tiempo) afectan la decisión de abandonar el servicio, utilizaremos gráficos de densidad (`sns.kdeplot`) y diagramas de caja (`sns.boxplot`).

Estos gráficos nos permitirán ver si las personas que se van tienden a tener facturas más altas o menos tiempo en la empresa.

In [ ]:
print("--- EVASIÓN POR FACTORES NUMÉRICOS (TIEMPO Y DINERO) ---")

# Variables numéricas continuas a analizar
columnas_num = ['Meses_Permanencia', 'Cargo_Mensual', 'Cargo_Total']
nombres_titulos_num = ['Meses en la Empresa (Tenure)', 'Cargo Mensual ($)', 'Cargos Totales Acumulados ($)']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Distribución de Variables Numéricas vs Evasión", fontsize=16, fontweight='bold', y=1.05)

for i, col in enumerate(columnas_num):
    # Usamos KDE Plot (Gráfico de Densidad) para ver donde se agrupan las masas de clientes
    sns.kdeplot(data=df, x=col, hue='Evasion', fill=True, palette=colores, ax=axes[i], alpha=0.5)
    axes[i].set_title(nombres_titulos_num[i], fontsize=13)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Densidad (Proporción de clientes)")

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico complementario de Cajas (Boxplots) para identificar valores atípicos (outliers) y medianas exactas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(columnas_num):
    sns.boxplot(data=df, x='Evasion', y=col, palette=colores, ax=axes[i])
    axes[i].set_title(f"Distribución exacta: {nombres_titulos_num[i]}", fontsize=12)
    axes[i].set_xlabel("Evasión (0=No, 1=Sí)")
    axes[i].set_ylabel("")

plt.tight_layout()
plt.show()

### 💡 Análisis Estratégico Dinero y Tiempo:
1. **Tiempo en la empresa (`Meses_Permanencia`)**: La montaña roja en el primer gráfico de densidad (KDE) está fuertemente sesgada a la izquierda. Esto es un grito de auxilio: **la inmensa mayoría de las deserciones ocurren en los primeros 5 meses**. El cliente nuevo es el más vulnerable. El Boxplot confirma que el 75% de los que se van, lo hacen antes de los 30 meses.
2. **Cargos Mensuales (`Cargo_Mensual`)**: Observa cómo la densidad roja sube brutalmente entre los \$70 y \$100 dólares. Los clientes con facturas más altas son, irónicamente, los que más abandonan el servicio. En el Boxplot, la mediana (línea central) de los que se van es muy superior a la de los que se quedan (
~$80 vs ~$60).
3. **Cargos Totales (`Cargo_Total`)**: La evasión es altísima al inicio porque se cruza con los meses de permanencia (meses bajos = cargos totales bajos). La gente se va antes de acumular mucho gasto histórico en la empresa.

## 11. EXTRA: Análisis de Correlación y Servicios Contratados

Para finalizar el modelado descriptivo, vamos a explorar las correlaciones matemáticas directas usando `df.corr()` y evaluaremos si acumular servicios en un mismo cliente (efecto ecosistema) funciona como un ancla para evitar el Churn.

In [ ]:
print("--- CÁLCULO DE SERVICIOS TOTALES ---")

# Contaremos cuántos servicios tiene cada cliente en total.
servicios = ['Servicio_Telefonico', 'Multiples_Lineas', 'Servicio_Internet', 'Seguridad_Online', 
             'Respaldo_Online', 'Proteccion_Dispositivo', 'Soporte_Tecnico', 'TV_Streaming', 'Peliculas_Streaming']

df['Total_Servicios'] = 0

for col in servicios:
    if col in df.columns:
        # Algunos son binarios (1/0) y otros textos ('Yes', 'Fiber optic', 'DSL').
        # Sumamos 1 si el cliente lo tiene activo.
        if df[col].dtype == object:
            # Buscar coincidencias positivas
            df['Total_Servicios'] += df[col].astype(str).str.contains('Yes|yes|Fiber|DSL', regex=True).astype(int)
        else:
            # Si es int/float binario, lo sumamos directamente
            df['Total_Servicios'] += df[col]

print("Distribución de Cantidad de Servicios Activos por Cliente:")
display(df['Total_Servicios'].value_counts().sort_index())

In [ ]:
print("--- MATRIZ DE CORRELACIÓN DE PEARSON ---")

# Seleccionamos variables numéricas clave para buscar relaciones con la Evasión
cols_corr = ['Evasion', 'Meses_Permanencia', 'Cargo_Mensual', 'Cargo_Total', 'Cuentas_Diarias', 'Total_Servicios', 'Adulto_Mayor']

# Calculamos la matriz
matriz_corr = df[cols_corr].corr()

# Heatmap con Seaborn
plt.figure(figsize=(10, 8))
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Mapa de Calor de Correlaciones (Heatmap)", fontsize=16)
plt.show()

In [ ]:
print("--- RELACIÓN: EVASIÓN VS ECOSISTEMA DE SERVICIOS ---")

plt.figure(figsize=(10, 5))
# barplot nos da el promedio de Evasión por cada grupo
sns.barplot(data=df, x='Total_Servicios', y='Evasion', color='#e74c3c')
plt.title("Tasa de Evasión Promedio según Cantidad Total de Servicios Contratados", fontsize=14)
plt.xlabel("Total de Servicios Adicionales Activos en el Hogar/Móvil", fontsize=12)
plt.ylabel("Tasa Promedio de Evasión (0 a 1)", fontsize=12)

# Añadir una línea negra punteada indicando el promedio global para comparar
tasa_global = df['Evasion'].mean()
plt.axhline(tasa_global, color='black', linestyle='--', label=f'Media Global ({round(tasa_global*100, 1)}%)')
plt.legend()
plt.show()

### 💡 Insights Predictivos Extra:
1. **Antagonismo Tiempo-Retención (Heatmap):** Fíjate cómo `Meses_Permanencia` alberga la correlación negativa más fuerte con `Evasion` (aprox -0.35). Es decir, a mayor cantidad de meses frente a la empresa, el valor numérico de la evasión baja diametralmente.
2. **`Cargo_Mensual` vs `Cuentas_Diarias` (Heatmap):** Su correlación es del 1.00 perfecta. Esto tiene todo el sentido lógico ya que una es la derivada matemática exacta de la otra (dividida entre 30).
3. **El brutal poder del Ecosistema (Gráfico de Barras):** Aquí yace el santo grial de las telecomunicaciones (Efecto Lock-in). Los clientes que contratan apenas 1 o 2 servicios resienten una tasa de abandono escandalosa (¡rozando el 40%!). Sin embargo, cuando logramos venderles 6, 7 o más servicios (Fibra + Telefonía + Soporte + TV), la fricción mental y técnica de cambiar de empresa de telecomunicaciones es tan alta que la deserción se desploma a niveles casi nulos (< 5%). **Esta es la campaña de fidelidad perfecta.**

---

# 📊 INFORME FINAL: Análisis de Evasión de Clientes (Churn) - Telecom X

## 🔹 Introducción
El presente análisis tiene como objetivo principal comprender y mitigar la **evasión de clientes (Churn)** en la empresa de telecomunicaciones Telecom X. El Churn representa la pérdida de suscriptores, lo cual impacta directamente en los ingresos de la compañía. Al identificar los factores clave que llevan a un cliente a cancelar su servicio, la empresa puede diseñar estrategias de retención proactivas y personalizadas.

## 🔹 Limpieza y Tratamiento de Datos
Para garantizar la calidad del análisis, se llevó a cabo un riguroso proceso de preparación de los datos extraídos de la API:
1. **Extracción y Aplanado:** Se importó el JSON crudo y se utilizó `pd.json_normalize` para convertir las estructuras anidadas en un DataFrame tabular.
2. **Corrección de Formatos:** Se detectó que la variable `Cargo_Total` estaba codificada como texto debido a espacios en blanco en cuentas nuevas. Se reemplazaron por `0.0` y se convirtió a numérico (`float64`).
3. **Eliminación de Valores Faltantes:** Se descubrieron 224 registros sin información en la variable objetivo (`Evasion`). Dado que no poseían valor predictivo, fueron eliminados.
4. **Estandarización y Binarización:** Se renombraron las columnas al español para mayor claridad y se transformaron variables lógicas afirmativas/negativas ('Yes'/'No') a un formato numérico binario (1/0), facilitando el procesamiento estadístico y de Machine Learning.

## 🔹 Análisis Exploratorio de Datos (EDA)
Durante la exploración, se identificaron varios hallazgos significativos mediante visualizaciones (Countplots, KDE Plots y Boxplots):
- **Desbalance de Clases:** La tasa de evasión global se sitúa en un **26.54%**.
- **Contratos:** La inmensa mayoría de las cancelaciones ocurren bajo el régimen de contrato **mes a mes**.
- **Tiempo en la Empresa (Tenure):** La densidad de abandono es críticamente alta durante los **primeros 5 meses**.
- **Precios:** Los clientes con cargos mensuales más altos (alrededor de los \$80) presentan una mayor tendencia a cancelar el servicio en comparación con los que pagan tarifas menores o básicas.
- **Servicio de Internet:** El servicio de **Fibra Óptica** presenta una anomalía con altas tasas de abandono frente a otras tecnologías como el ADSL.

## 🔹 Conclusiones e Insights
El análisis revela que la evasión en Telecom X no es aleatoria, sino que está fuertemente correlacionada con la falta de compromiso a largo plazo (contratos mensuales), costos elevados percibidos en etapas tempranas (primeros 5 meses) y fricción o insatisfacción específica en servicios premium como la Fibra Óptica.

La vulnerabilidad ocurre al inicio de ciclo de vida del cliente. Si un cliente sobrevive la barrera del primer año y/o migra a contratos anuales, su probabilidad de abandono se desploma drásticamente.

## 🔹 Recomendaciones Estratégicas
1. **Incentivos para Contratos Anuales:** Crear ofertas introductorias o descuentos agresivos a cambio de firmar contratos de 1 o 2 años para clientes nuevos, con el fin de superar la barrera crítica de los primeros meses.
2. **Revisión del Servicio de Fibra Óptica:** Iniciar de inmediato una auditoría técnica y de precios sobre el producto de Fibra Óptica. Su alta tasa de abandono sugiere problemas de calidad o una competencia con mejores precios.
3. **Programa de Onboarding y Fidelización Temprana:** Implementar un seguimiento especializado y soporte técnico premium gratuito durante los primeros 6 meses de servicio (donde ocurre la mayor tasa de deserción) para mejorar la experiencia.
4. **Automatización de Pagos:** Fomentar el uso de transferencias y pagos automáticos frente al cheque electrónico, ya que los primeros reducen la fricción psicológica de pagar mes a mes.
